## Codigo para pruebas de solo encoder para prediccion de elo

In [1]:
#El MAE te dirá "en promedio, el modelo se equivoca por X puntos de Elo"

import chess
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import chess.pgn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io
from tqdm.notebook import tqdm
from scipy.stats import spearmanr
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
import random
from torch.utils.data import IterableDataset, DataLoader
import joblib
import torch
SEED = 37
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

tf.keras.utils.set_random_seed(SEED)

plt.ion()

I0000 00:00:1789000740.925333   32248 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789000743.622646   32248 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# datos tomados del train
ELO_MIN = 900
ELO_MAX = 2750
ELO_MEAN = 1551.1 
ELO_STD = 299.6  
ELO_MEDIAN = 1548.0
ELO_Q1 = 1332.0
ELO_Q3 = 1763.0
ELO_IQR = ELO_Q3 - ELO_Q1

In [4]:
def normalizar_elo(elo, metodo):
    """Aplica la normalización elegida."""
    if metodo == "minmax":
        # Rango [0, 1]
        return (elo - ELO_MIN) / (ELO_MAX - ELO_MIN)
    elif metodo == "standard":
        # Centrado en 0, desvío 1 (Z-score)
        return (elo - ELO_MEAN) / ELO_STD
    elif metodo == "robust":
        # Resistente a valores atípicos (outliers)
        return (elo - ELO_MEDIAN) / ELO_IQR
    else: # "none" o error
        return elo

In [5]:
def board_to_bitboard_vector_13(board):
    """Convierte un chess.Board a un vector binario de 832 bits (13 x 64)."""
    piece_types = [
        (chess.PAWN, chess.WHITE), (chess.PAWN, chess.BLACK),
        (chess.ROOK, chess.WHITE), (chess.ROOK, chess.BLACK),
        (chess.KNIGHT, chess.WHITE), (chess.KNIGHT, chess.BLACK),
        (chess.BISHOP, chess.WHITE), (chess.BISHOP, chess.BLACK),
        (chess.QUEEN, chess.WHITE), (chess.QUEEN, chess.BLACK),
        (chess.KING, chess.WHITE), (chess.KING, chess.BLACK),
    ]

    vec = np.zeros(13 * 64, dtype=np.uint8)

    # Llenamos los 12 primeros bitboards
    for i, (piece, color) in enumerate(piece_types):
        bitboard = board.pieces(piece, color)
        for square in bitboard:
            vec[i * 64 + square] = 1

    # Bitboard 13: casillas vacías
    occupied = board.occupied
    for square in chess.SQUARES:
        if not occupied & chess.BB_SQUARES[square]:
            vec[12 * 64 + square] = 1

    return vec

In [6]:
import tensorflow as tf
import numpy as np
import chess.pgn

def pgn_generator(pgn_filename, ply_numbers, norm_type, flatten):
    def generator():
        with open(pgn_filename, "r", encoding="utf-8") as f:
            while True:
                try:
                    # 1. Leemos la partida. Si algo falla aquí, va al except.
                    game = chess.pgn.read_game(f)
                    if game is None:
                        break # Fin del archivo
                    
                    board = game.board()
                    secuencia = []
                    max_ply = max(ply_numbers)
                    
                    for i, move in enumerate(game.mainline_moves()):
                        board.push(move)
                        ply = i + 1
                        
                        if ply in ply_numbers:
                            secuencia.append(board_to_bitboard_vector_13(board))
                            
                        if ply >= max_ply:
                            break
                    
                    if len(secuencia) == len(ply_numbers):
                        # Intentamos obtener el Elo
                        elo = (
                            int(game.headers.get("WhiteElo", 0)) +
                            int(game.headers.get("BlackElo", 0))
                        )/2
                        
                        elo_norm = normalizar_elo(elo, norm_type)
                        
                        # 2. Forzamos estrictamente los tipos a float32
                        x = np.array(secuencia, dtype=np.float32)
                        
                        if flatten == 1:
                            x = x.flatten()
                            
                        # CLAVE: Convertimos el escalar de Python a un float32 de NumPy
                        yield x, np.float32(elo_norm)
                        
                except Exception as e:
                    # Si la partida está corrupta, un jugador no tiene Elo, o hay un bug en el PGN...
                    # Simplemente ignoramos la partida y pasamos a la siguiente.
                    continue
                    
    return generator

In [7]:
def crear_dataset(pgn_filename, ply_numbers, norm_type, flatten=1, batch_size=256):
    num_plies = len(ply_numbers)
    
    if flatten == 1:
        input_shape = (num_plies * 832,) 
    else:
        input_shape = (num_plies, 832)
    
    dataset = tf.data.Dataset.from_generator(
        pgn_generator(pgn_filename, ply_numbers, norm_type, flatten),
        output_signature=(
            tf.TensorSpec(shape=input_shape, dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )
    return dataset.batch(batch_size).repeat().prefetch(tf.data.AUTOTUNE)


In [12]:
# Instanciamos los datasets
tipo_norm ="standard" #"standard", "minmax", "robust"
flatten = 1
ply_numbers = [20]
batch_size=256
train_dataset = crear_dataset("../data/generado/noleak/train.pgn", ply_numbers,tipo_norm,flatten = flatten, batch_size=batch_size) 
val_dataset = crear_dataset("../data/generado/noleak/val.pgn", ply_numbers,tipo_norm, flatten = flatten,batch_size=batch_size)
test_dataset = crear_dataset("../data/generado/noleak/test.pgn", ply_numbers,tipo_norm, flatten = flatten,batch_size=batch_size)

In [9]:
if flatten == 1:
    input_shape = (len(ply_numbers) * 832,) 
else:
    input_shape = (len(ply_numbers), 832)

In [10]:
encoder_salida = f"../data/encoders/only_enc/ply20.npy"
archivo_salida_loss = f"../data/losses/only_enc/ply20.npy"

In [11]:
# === DEFINICIÓN DEL ENCODER===

input_layer = keras.Input(shape=(input_shape))
encoded = keras.layers.Dense(512, activation="relu")(input_layer)
encoded = keras.layers.Dense(256, activation="relu")(encoded)
encoded = keras.layers.Dense(128, activation="relu")(encoded)
encoded = keras.layers.Dense(32, activation="relu")(encoded)
output_layer = keras.layers.Dense(1)(encoded)


encoder = keras.Model(input_layer, output_layer)
encoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

In [14]:
# entrenamiento
history = encoder.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    steps_per_epoch=69886//batch_size,
    validation_steps=15201//batch_size,
    callbacks=[
        keras.callbacks.EarlyStopping( # frena si deja de mejorar
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )
    ]
)
joblib.dump(encoder, encoder_salida) #guardar el encoder


Epoch 1/50


/home/agust/tesis/venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


272/272 ━━━━━━━━━━━━━━━━━━━━ 172s 606ms/step - loss: 0.7418 - mae: 0.6890 - val_loss: 0.7054 - val_mae: 0.6724
Epoch 2/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 164s 605ms/step - loss: 0.6757 - mae: 0.6549 - val_loss: 0.7006 - val_mae: 0.6656
Epoch 3/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 168s 619ms/step - loss: 0.6281 - mae: 0.6299 - val_loss: 0.7167 - val_mae: 0.6716
Epoch 4/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 175s 644ms/step - loss: 0.5783 - mae: 0.6029 - val_loss: 0.7283 - val_mae: 0.6759
Epoch 5/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 1837s 7s/step - loss: 0.5318 - mae: 0.5770 - val_loss: 0.7755 - val_mae: 0.6940


['../data/encoders/only_enc/ply20.npy']

In [15]:
def desnormalizar_elo(elo_norm, metodo):
    """Convierte el Elo normalizado de vuelta a puntos de Elo reales."""
    if metodo == "minmax":
        return elo_norm * (ELO_MAX - ELO_MIN) + ELO_MIN
    elif metodo == "standard":
        return elo_norm * ELO_STD + ELO_MEAN
    elif metodo == "robust":
        return elo_norm * ELO_IQR + ELO_MEDIAN
    else:
        return elo_norm

In [ ]:
test_loss, test_mae = encoder.evaluate(test_dataset)
np.save(archivo_salida_loss, test_loss) # cambiar
print(f"MSE en Test: {test_loss:.4f}")
print(f"MAE en Test (normalizado): {test_mae:.4f}")

NameError: name 'test_ds' is not defined

In [17]:
y_true_list = []
y_pred_list = []

# Iteramos sobre el dataset de test lote por lote
for x_batch, y_batch in test_dataset:
    # predict_on_batch es muy rápido para esto
    predicciones = encoder.predict_on_batch(x_batch)
    
    # Guardamos los valores reales y las predicciones
    y_true_list.extend(y_batch.numpy())
    y_pred_list.extend(predicciones.flatten())

y_true_norm = np.array(y_true_list)
y_pred_norm = np.array(y_pred_list)

# --- DESNORMALIZAMOS PARA EL GRÁFICO ---
elos_reales = desnormalizar_elo(y_true_norm, tipo_norm)
elos_predichos = desnormalizar_elo(y_pred_norm, tipo_norm)

# Calculamos el Error Absoluto (MAE) por partida en "Puntos de Elo"
error_absoluto = np.abs(elos_reales - elos_predichos)

KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(elos_reales, elos_predichos, alpha=0.3)

# Diagonal ideal: predicción = Elo real
min_elo = min(elos_reales.min(), elos_predichos.min())
max_elo = max(elos_reales.max(), elos_predichos.max())

plt.plot(
    [min_elo, max_elo],
    [min_elo, max_elo],
    linestyle="--"
)

plt.xlabel("Elo real")
plt.ylabel("Elo predicho")
plt.title("Elo real vs Elo predicho")

plt.xlim(min_elo, max_elo)
plt.ylim(min_elo, max_elo)

plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
error = elos_predichos - elos_reales

plt.figure(figsize=(8, 5))

plt.scatter(elos_reales, error, alpha=0.3)

plt.axhline(0, linestyle="--")

plt.xlabel("Elo real")
plt.ylabel("Error (Elo predicho - Elo real)")
plt.title("Error de predicción según Elo")

plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 1. Agrupamos los datos en un DataFrame para manejarlos fácil
df_resultados = pd.DataFrame({
    'Elo_Real': elos_reales,
    'Error': error_absoluto
})

# 2. Creamos rangos de Elo (ej. de 100 en 100)
bins = np.arange(800, 3100, 100)
df_resultados['Elo_Bin'] = pd.cut(df_resultados['Elo_Real'], bins=bins)

# 3. Calculamos el Error Promedio para cada rango de Elo
error_por_rango = df_resultados.groupby('Elo_Bin')['Error'].mean()

# Extraemos el centro de cada rango para el eje X del gráfico (ej. 1050 para el bin 1000-1100)
x_centers = [bin.mid for bin in error_por_rango.index]
y_errores = error_por_rango.values

# 4. Dibujamos el gráfico
plt.figure(figsize=(10, 6))

# Dibujamos la línea de tendencia de errores
plt.plot(x_centers, y_errores, marker='o', linestyle='-', color='b', linewidth=2, label='Error Medio (MAE)')

# Añadimos un scatter plot de fondo con mucha transparencia (opcional, para ver la dispersión)
plt.scatter(elos_reales, error_absoluto, alpha=0.02, color='gray', label='Partidas individuales')

plt.title('Error de Predicción del Modelo vs. Nivel de Elo', fontsize=14)
plt.xlabel('Elo Real de la Partida', fontsize=12)
plt.ylabel('Error Absoluto (Puntos de Elo)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

# Guardamos el gráfico
plt.savefig("C:/Users/agust/Desktop/facu temporal/tesis/generado/error_vs_elo.png", dpi=300)
plt.show()